# 📊 Mini-Project A: Virtual Joint Analysis Dashboard

**Role:** Data Science Lead  
**Objective:** Verify that the motor (virtual or real) meets the "Truth" requirements before we allow it to proceed to the next phase.

### 🔍 What we are looking for:
1.  **Tracking Accuracy:** Does the `Actual Position` (Blue) closely follow the `Target Position` (Green)?
2.  **Steady-State Error:** When the motor stops, is the error close to 0°?
3.  **Noise Levels:** Is the sensor data jittering too much?

## 📈 Performance Visualizations

The following plots visualize the "Data Contract" we established with the Hardware Lead.

### 1. Position Tracking
* **Green Dashed Line:** The perfect command (what we *wanted* to happen).
* **Blue Line:** The reality (what *actually* happened).
* **Lag:** Notice the slight delay between the green and blue lines? That is the simulated inertia.

### 2. Error Analysis (The "Verdict")
* **Red Line:** The difference between Target and Actual.
* **Thresholds:** The red dotted lines represent our safety limits (+/- 1.0 degree).

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import re

BASE_LOG_DIR = os.path.join("..", "..", "logs")
MAX_ALLOWED_ERROR = 1.0

# List all run folders that match 'run' + number
run_folders = [
    f.path for f in os.scandir(BASE_LOG_DIR) 
    if f.is_dir() and re.match(r"run[_-]?\d+", f.name, re.IGNORECASE)
]

# Sort by the numeric run number
run_folders = sorted(
    run_folders, 
    key=lambda x: int(re.search(r"(\d+)", os.path.basename(x)).group(1))
)

if not run_folders:
    print("❌ No run folders found!")
    exit()

import ipywidgets as widgets
from IPython.display import display, clear_output

def load_run(run_index):
    clear_output(wait=True)
    selected_run = run_folders[run_index]
    print(f"✅ Selected run: {os.path.basename(selected_run)}")

    # Load CSVs from that run
    log_files = glob.glob(os.path.join(selected_run, "*.csv"))
    if log_files:
        print(f"✅ Loaded {len(log_files)} file(s) from run: {os.path.basename(selected_run)}")
    else:
        print("❌ No log files found in selected run!")
        return

    # --- Plotting + summary logic (same as before) ---
    summary = []
    for log_file in log_files:
        df = pd.read_csv(log_file)
        df['time_sec'] = df['timestamp_ms'] / 1000.0
        hz = df['hz'].iloc[0] if 'hz' in df.columns else "?"
        speed = df['velocity_cmd'].iloc[0] if 'velocity_cmd' in df.columns else "?"
        max_error = df['error'].abs().max()
        avg_error = df['error'].abs().mean()
        verdict = "PASS" if max_error <= MAX_ALLOWED_ERROR else "FAIL"

        summary.append({
            "file": os.path.basename(log_file),
            "hz": hz,
            "speed_deg_s": speed,
            "max_error": round(max_error, 3),
            "avg_error": round(avg_error, 3),
            "verdict": verdict
        })

        # --- Plotting ---
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        fig.suptitle(f"{os.path.basename(log_file)} | hz={hz}, speed={speed} | {verdict}")
        ax1.plot(df['time_sec'], df['target_pos'], 'g--', label='Target')
        ax1.plot(df['time_sec'], df['actual_pos'], 'b-', label='Actual', alpha=0.75)
        ax1.set_ylabel("Position (deg)")
        ax1.legend()
        ax1.grid(True)
        ax2.plot(df['time_sec'], df['error'], 'r-')
        ax2.axhline(MAX_ALLOWED_ERROR, linestyle=':', color='darkred')
        ax2.axhline(-MAX_ALLOWED_ERROR, linestyle=':', color='darkred')
        ax2.set_ylabel("Error (deg)")
        ax2.set_xlabel("Time (s)")
        ax2.grid(True)
        plt.tight_layout()
        plt.show()

    # --- Summary Table ---
    summary_df = pd.DataFrame(summary)
    print("\n" + "=" * 80)
    print("📊 FULL SIMULATION SUMMARY")
    print("=" * 80)
    print(summary_df.to_string(index=False))
    print("=" * 80)

    passes = summary_df[summary_df['verdict'] == "PASS"]
    fails = summary_df[summary_df['verdict'] == "FAIL"]
    if not passes.empty:
        print("\n✅ SAFE OPERATING ENVELOPE (Observed):")
        print(passes[['hz', 'speed_deg_s']].drop_duplicates().to_string(index=False))
    if not fails.empty:
        print("\n❌ FAILED REGIMES (Avoid or Fix):")
        print(fails[['hz', 'speed_deg_s']].drop_duplicates().to_string(index=False))


# --- Dropdown ---
run_dropdown = widgets.Dropdown(
    options=[(os.path.basename(f), i) for i, f in enumerate(run_folders)],
    value=len(run_folders)-1,  # default to newest run
    description='Select Run:'
)
widgets.interact(load_run, run_index=run_dropdown)

interactive(children=(Dropdown(description='Select Run:', index=2, options=(('run1', 0), ('run2', 1), ('run5',…

<function __main__.load_run(run_index)>